In [ ]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import OpenAIEmbeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings

C:\Windows\Temp\ipykernel_1080\1109008249.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Step 0: Load the pdf into text format

In [2]:
text_data = PyPDFLoader("NovaS.pdf").load()

<!--  -->

In [3]:
for page in text_data:
    page.metadata["source"]="NovaSX.pdf"

text_data

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-31T11:24:15-03:00', 'author': 'Ansh Lamba', 'moddate': '2026-03-31T11:24:15-03:00', 'source': 'NovaSX.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future. At the \nbeginning, the company did not have large investments or a big office. Instead, it started \nwith only six employees working together in a small shared workspace. The founders were \nnot focused on becoming successful overnight. Their main goal was to build strong \ntechnical knowledge, gain practical experience, and slowly grow by 

STEP-1: Creating Chunks of the text data

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)

chunks = splitter.split_documents(text_data)
len(chunks)

101

Step-2&3: Creating Embeddings and Storing them in Vector DB

In [5]:
# embed_model = OpenAIEmbeddings(model="text-embedding-3-small")
embed_model = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
embedded_chunks = embed_model.embed_documents([i.page_content for i in chunks])
embedded_chunks[0]

[-0.0071337917,
 -0.0024039722,
 0.015069621,
 -0.09315443,
 0.0039596176,
 0.010406283,
 -0.003551804,
 -0.00822853,
 0.0039925994,
 0.011684152,
 -0.026310448,
 -0.009637208,
 0.0020898632,
 0.05070348,
 0.10534179,
 0.012014975,
 -0.015993143,
 -0.0005892312,
 0.0041541685,
 -0.0030058657,
 -0.0035432393,
 0.007976135,
 0.026032327,
 -0.017698037,
 -0.0047360933,
 -0.023649663,
 0.019695798,
 0.0015269268,
 0.0047064745,
 0.01155726,
 -0.0029810732,
 0.013077376,
 0.008564928,
 0.026106179,
 0.001973216,
 0.012165739,
 0.0010474339,
 -0.020195317,
 -0.018899756,
 -0.0034067538,
 0.011898069,
 -0.008350914,
 -0.034809347,
 -0.029872317,
 -0.00736157,
 0.01127585,
 0.0042976993,
 0.003229004,
 0.0030710287,
 0.011381783,
 0.015889527,
 -0.006317377,
 -0.0282376,
 -0.21231619,
 -0.009046265,
 -0.009201602,
 -0.007467311,
 -0.009855643,
 -0.005964542,
 0.007707912,
 -0.0032807367,
 0.012736169,
 -0.016900886,
 -0.023082113,
 0.009394612,
 -0.02025343,
 -0.001158242,
 -0.0030059547,
 -0.

In [6]:
from langchain_community.vectorstores import Chroma

In [10]:
# embed_model = OpenAIEmbeddings(model="text-embedding-3-small")
embed_model = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
   

In [11]:
chroma_db = Chroma.from_documents(chunks, embed_model, persist_directory="./chroma_db")

Step - 4 : Connection & Retrieval

In [14]:
chroma_db_con = Chroma(persist_directory="./chroma_db", embedding_function=embed_model)

In [15]:
chroma_db_con.similarity_search("When was NovaSphere organsiation founded?", k=3)

[Document(metadata={'creator': 'Microsoft® Word for Microsoft 365', 'producer': 'Microsoft® Word for Microsoft 365', 'total_pages': 3, 'source': 'NovaSX.pdf', 'moddate': '2026-03-31T11:24:15-03:00', 'page_label': '3', 'author': 'Ansh Lamba', 'creationdate': '2026-03-31T11:24:15-03:00', 'page': 2}, page_content='Today, NovaSphere Technologies is considered a reliable organization that provides data'),
 Document(metadata={'creationdate': '2026-03-31T11:24:15-03:00', 'total_pages': 3, 'producer': 'Microsoft® Word for Microsoft 365', 'page': 1, 'creator': 'Microsoft® Word for Microsoft 365', 'source': 'NovaSX.pdf', 'author': 'Ansh Lamba', 'page_label': '2', 'moddate': '2026-03-31T11:24:15-03:00'}, page_content='The year 2020 was difficult for many businesses around the world, but NovaSphere'),
 Document(metadata={'source': 'NovaSX.pdf', 'creationdate': '2026-03-31T11:24:15-03:00', 'page_label': '2', 'author': 'Ansh Lamba', 'producer': 'Microsoft® Word for Microsoft 365', 'moddate': '2026-0

In [16]:
chroma_db_con.similarity_search("By 2019, how many employees were there?", k=3)

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'page': 0, 'total_pages': 3, 'author': 'Ansh Lamba', 'source': 'NovaSX.pdf', 'moddate': '2026-03-31T11:24:15-03:00', 'creationdate': '2026-03-31T11:24:15-03:00', 'page_label': '1'}, page_content='By the beginning of 2019, the organization had grown to more than fifteen employees. This'),
 Document(metadata={'total_pages': 3, 'source': 'NovaSX.pdf', 'author': 'Ansh Lamba', 'page': 1, 'creator': 'Microsoft® Word for Microsoft 365', 'moddate': '2026-03-31T11:24:15-03:00', 'producer': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-31T11:24:15-03:00', 'page_label': '2'}, page_content='number of employees increased again, and the company also started hiring people who'),
 Document(metadata={'source': 'NovaSX.pdf', 'page_label': '2', 'producer': 'Microsoft® Word for Microsoft 365', 'author': 'Ansh Lamba', 'page': 1, 'creationdate': '2026-03-31T11:24:15-03:00', 'm

Step 5: LLM Integration and Answer Generation

In [ ]:
# llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")
# llm.invoke("what is the name of the company?")


In [ ]:
user_query = input("Enter your question: ")
# user_query = "By 2019, how many employees were there?"
rel_chunks = chroma_db_con.similarity_search(user_query, k=3)

rel_chunks_content = []
for i, chunk in enumerate(rel_chunks):
    rel_chunks_content.append(chunk.page_content)

# str(rel_chunks_content)

llm.invoke(f"{user_query}, Use the following content to answer the question: {str(rel_chunks_content)}")